# JAXOOM T4 allocator-residency refinement

This notebook refines the seven existing T4 thresholds, then screens and probes targeted additional candidates. It stops model fitting unless the evidence gate is met.

In [ ]:
import json, sys, importlib, zipfile, collections, statistics
from pathlib import Path
REPO = Path("/content/jaxoom")
PINNED_COMMIT = "aecb9ec8743b724a832e1c8a02904d528596edc1"


## Install and pin the campaign infrastructure

In [ ]:
!pip install -q "jax[cuda12]==0.11.0"
!rm -rf /content/jaxoom
!git clone -q https://github.com/Slavov88/jaxoom.git /content/jaxoom
!cd /content/jaxoom && git checkout -q aecb9ec8743b724a832e1c8a02904d528596edc1 && pip install -q -e .


## Verify Tesla T4 and JAX versions

In [ ]:
sys.path.insert(0, "/content/jaxoom/src")
sys.modules.pop("jaxoom", None)
importlib.invalidate_caches()
import jax, jaxlib, jaxoom
assert jax.__version__ == "0.11.0", jax.__version__
assert jaxlib.__version__ == "0.11.0", jaxlib.__version__
assert jax.default_backend() == "gpu", jax.default_backend()
device = jax.devices()[0]
assert "T4" in getattr(device, "device_kind", str(device)), device
environment = {"jax_version":jax.__version__,"jaxlib_version":jaxlib.__version__,"backend":jax.default_backend(),"device":str(device),"device_kind":getattr(device,"device_kind",None),"pinned_commit":PINNED_COMMIT,"preallocate":False}
Path("/content/environment.json").write_text(json.dumps(environment,indent=2)+"
")
print(json.dumps(environment,indent=2))


## Refine the existing seven useful thresholds

In [ ]:
!cd /content/jaxoom && PYTHONPATH=src:experiments python experiments/t4_allocator_residency_refinement.py --thresholds experiments/allocator_residency_t4_thresholds_2026-09-09.json --raw experiments/allocator_residency_t4_raw_2026-09-09.json --fraction-map experiments/allocator_residency_t4_fraction_map_2026-09-09.json --output /content/refinement_raw.json --timeout 60 --max-probes 35 --target-width 67108864


## Screen moderate additional candidates

In [ ]:
!cd /content/jaxoom && PYTHONPATH=src:experiments JAX_PLATFORMS=cpu python experiments/allocator_residency_campaign.py --screening /content/screening.json --thresholds /content/unused.json --screen-only


In [ ]:
screening = json.loads(Path("/content/screening.json").read_text())["rows"]
old = {row["configuration_id"] for row in json.loads((REPO / "experiments/allocator_residency_t4_thresholds_2026-09-09.json").read_text())["rows"]}
new = [row for row in screening if row.get("status") == "THRESHOLD_CANDIDATE" and row["configuration_id"] not in old]
# Bound the additional campaign and prioritize non-attention families.
new.sort(key=lambda row: (row["family"] == "attention", row.get("calibrated_upper_bytes", 0)))
new = new[:12]
Path("/content/new_manifest.json").write_text(json.dumps({"status":"MANIFEST","candidates":new},indent=2)+"
")
print(len(new), collections.Counter(row["family"] for row in new))


## Probe additional candidates with checkpointing

In [ ]:
!cd /content/jaxoom && PYTHONPATH=src:experiments python experiments/t4_allocator_residency.py --manifest /content/new_manifest.json --fraction-map experiments/allocator_residency_t4_fraction_map_2026-09-09.json --output /content/new_raw.json --timeout 60 --max-probes 60


## Derive thresholds, transfer diagnostics, and model gate

In [ ]:
def outcome(row):
    return (row.get("execution_statuses") or [row.get("status") or row.get("compile_status") or "OTHER_FAILURE"])[0]
def cap(row):
    return ((row.get("snapshots") or [{}])[0]).get("allocator_limit_bytes")
def derive(rows):
    groups = {}
    for row in rows: groups.setdefault(row.get("configuration_id"), []).append(row)
    result=[]
    for cid, group in groups.items():
        fits=[r for r in group if outcome(r)=="FIT" and cap(r) is not None]
        ooms=[r for r in group if outcome(r) in {"COMPILE_OOM","EXECUTION_OOM"} and cap(r) is not None]
        if fits and ooms:
            lo=max(cap(r) for r in ooms); hi=min(cap(r) for r in fits)
            result.append({"configuration_id":cid,"family":group[0].get("family"),"configuration":group[0].get("configuration"),"dtype":group[0].get("dtype"),"known_oom_capacity":lo,"known_fit_capacity":hi,"bracket_width_bytes":hi-lo})
    return result
refined_raw=json.loads(Path("/content/refinement_raw.json").read_text()).get("rows",[])
new_raw=json.loads(Path("/content/new_raw.json").read_text()).get("rows",[])
refined=derive(refined_raw); new=derive(new_raw)
Path("/content/refined_thresholds.json").write_text(json.dumps({"status":"OBSERVED","rows":refined},indent=2)+"
")
Path("/content/new_thresholds.json").write_text(json.dumps({"status":"OBSERVED","rows":new},indent=2)+"
")
all_thresholds=refined+new
families=collections.Counter(row["family"] for row in all_thresholds)
dtypes=collections.Counter(row["dtype"] for row in all_thresholds)
summary={"status":"OBSERVED","refined_count":len(refined),"new_count":len(new),"total_useful_t4_thresholds":len(all_thresholds),"families":dict(families),"dtypes":dict(dtypes),"outcomes":dict(collections.Counter(outcome(row) for row in refined_raw+new_raw))}
gate=len(all_thresholds)>=12 and len(families)>=3
model={"status":"NOT_RUN","reason":"evidence gate not met" if not gate else "PENDING_OFFLINE_MODEL_COMPARISON","evidence_gate":{"threshold_count":len(all_thresholds),"family_count":len(families),"met":gate}}
Path("/content/summary.json").write_text(json.dumps(summary,indent=2)+"
")
Path("/content/model_comparison.json").write_text(json.dumps(model,indent=2)+"
")
# Compare conservative upper bounds with committed RTX labels.
rtx=json.loads((REPO/"experiments/allocator_residency_thresholds_2026-09-09.json").read_text()).get("useful_thresholds",[])
rtx={r["configuration_id"]:r for r in rtx}
paired=[]
for row in all_thresholds:
    if row["configuration_id"] in rtx:
        paired.append({"configuration_id":row["configuration_id"],"family":row["family"],"dtype":row["dtype"],"t4_upper":row["known_fit_capacity"],"rtx_upper":rtx[row["configuration_id"]]["required_allocator_upper_bytes"],"upper_ratio":row["known_fit_capacity"]/rtx[row["configuration_id"]]["required_allocator_upper_bytes"]})
Path("/content/paired_comparison.json").write_text(json.dumps({"status":"OBSERVED","rows":paired},indent=2)+"
")
Path("/content/raw_probes.json").write_text(json.dumps({"status":"OBSERVED","rows":refined_raw+new_raw},indent=2)+"\n")
print(json.dumps(summary,indent=2))


## Package checkpoint and results

In [ ]:
files=["environment.json","refined_thresholds.json","new_thresholds.json","raw_probes.json","screening.json","paired_comparison.json","summary.json","model_comparison.json"]
with zipfile.ZipFile("/content/jaxoom_t4_allocator_residency_refined.zip","w",compression=zipfile.ZIP_DEFLATED) as archive:
    for name in files: archive.write("/content/"+name,arcname=name)
from google.colab import files
files.download("/content/jaxoom_t4_allocator_residency_refined.zip")
